In [4]:
import pandas as pd

data = pd.read_json("datasets/alpaca_data_cleaned-dutch.jsonl", lines=True)

def merge_instruction_into_input(df, instruction_col="instruction", input_col="input"):
    df = df.copy()
    df[input_col] = (
        df[instruction_col].fillna("").str.strip()
        + "\ninput: "
        + df[input_col].fillna("").str.strip()
    )
    return df

# instructions and input in one column
data = merge_instruction_into_input(data)

In [5]:
def split_train_test(data, train_frac=0.8, random_state=42, reset_idx=True):
    train_data = data.sample(frac=train_frac, random_state=random_state)
    test_data = data.drop(train_data.index)

    if reset_idx:
        train_data = train_data.reset_index(drop=True)
        test_data = test_data.reset_index(drop=True)

    return train_data, test_data


train_data, test_data = split_train_test(data, train_frac=0.8, random_state=42)

print(f"Train size: {len(train_data)}")
print(f"Test size: {len(test_data)}")

# save train and test set - ensure consistency in sampling
train_data.to_json("datasets/alpaca_train.jsonl", orient="records", lines=True)
test_data.to_json("datasets/alpaca_test.jsonl", orient="records", lines=True)

Train size: 41370
Test size: 10342


In [ ]:
from finetuning import data
import json

with open("configs/qlora_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

model_name = config["model"]["name"]


from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

# load train set

hf_dataset = data.to_hf_dataset(train_data)

chat_template = """Beantwoord de volgende vraag zo goed mogelijk.

### Vraag:
{INPUT}

### Antwoord:
{OUTPUT}
"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

def formatting_prompts_func(data):
    inputs       = data["input"]
    outputs      = data["output"]
    texts = []
    for input, output in zip(inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = chat_template.format(INPUT=input, OUTPUT=output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }


formatted_train_data = hf_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=hf_dataset.column_names,)


# Save formatted dataset to disk
output_path = "datasets/alpaca_train_formatted"
formatted_train_data.save_to_disk(output_path)
print(f"Saved {len(formatted_train_data)} examples to {output_path}")
print(f"Sample:\n{formatted_train_data[0]['text'][:200]}...")

Saving the dataset (1/1 shards): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 41370/41370 [00:00<00:00, 70484.74 examples/s]


Saved 41370 examples to datasets/alpaca_train_formatted
Sample:
Beantwoord de volgende vraag zo goed mogelijk.

### Vraag:
Schrijf een gedicht van 7 regels met behulp van de gegeven woorden
input: paraplu, rail, water, lucht, gras, blad

### Antwoord:
Onder de don...


In [8]:
from datasets import load_from_disk

alpaca_train_formatted = load_from_disk(output_path, keep_in_memory=True)
print(alpaca_train_formatted)
print(alpaca_train_formatted[0]["text"][:200] + "...")

Dataset({
    features: ['text'],
    num_rows: 41370
})
Beantwoord de volgende vraag zo goed mogelijk.

### Vraag:
Schrijf een gedicht van 7 regels met behulp van de gegeven woorden
input: paraplu, rail, water, lucht, gras, blad

### Antwoord:
Onder de don...
